# Fast 2-parameter BBH PE — Eryn vs pocoMC

A Gaussian-noise BBH injection, then parameter estimation over **only the two intrinsic mass
parameters** (chirp mass $\mathcal{M}$ and mass ratio $q$) with the heavy-tailed **hyperbolic**
likelihood. The other 12 CBC parameters are held at their injected values, so this is a fast,
low-dimensional demonstration that runs in minutes. We run the *same* problem through both
samplers and compare the posteriors and wall-clock times.

The problem setup is reused from `examples/pe_fast/bbh_fast_pe.py` (a runnable script version of
this notebook).

In [1]:
import os, sys, time, warnings
import numpy as np
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import corner

sys.path.insert(0, os.path.abspath('examples/pe_fast'))
from bbh_fast_pe import build_problem, make_priors
from hyperwave.inference import LVKinference

OUT = 'results/pe_fast_nb'
os.makedirs(os.path.join(OUT, 'chains'), exist_ok=True)
NSEG = 2
loglike_2d, theta_true, n_noise, info = build_problem(nsegs=NSEG, seed=42)
priors, noise_priors = make_priors(NSEG)
print(f'injected: chirp_mass={theta_true[0]:.3f}  mass_ratio={theta_true[1]:.3f}')
print(f'sampled dims: 2 science + {n_noise} hyperbolic noise-shape')

injected: chirp_mass=28.096  mass_ratio=0.806
sampled dims: 2 science + 3 hyperbolic noise-shape


## 1. Run Eryn (parallel-tempered ensemble MCMC)

In [2]:
def run(sampler, kw, tag):
    h5 = os.path.join(OUT, 'chains', f'{tag}_eryn.h5')
    if os.path.exists(h5):
        os.remove(h5)
    t0 = time.perf_counter()
    inf = LVKinference(loglike_2d, sampler_name=sampler, priors=priors, noise_priors=noise_priors,
                       common_params={'save_dir': OUT, 'TAG': tag, 'like': 'hyperbolic'},
                       sampler_kwargs=kw)
    inf.run()
    return time.perf_counter() - t0, inf.get_samples()

t_eryn, s_eryn = run('eryn', dict(nwalkers=20, ntemps=8, burn=2000, nsteps=5000), 'fast_eryn')
print(f'Eryn: {t_eryn:.1f} s | {s_eryn.shape[0]} samples | '
      f'Mc={np.median(s_eryn[:,0]):.3f}  q={np.median(s_eryn[:,1]):.3f}')

> Running Eryn MCMC...


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 1/2000 [00:00<05:54,  5.63it/s]

  0%|          | 2/2000 [00:00<06:04,  5.48it/s]

  0%|          | 3/2000 [00:00<06:36,  5.03it/s]

...

100%|█████████▉| 4999/5000 [26:56<00:00,  3.35it/s]

100%|██████████| 5000/5000 [26:57<00:00,  3.38it/s]

100%|██████████| 5000/5000 [26:57<00:00,  3.09it/s]

> Eryn sampling finished. Output at: results/pe_fast_nb/chains/fast_eryn_eryn.h5
Loading samples from results/pe_fast_nb/chains/fast_eryn_eryn.h5


50000
Eryn: 2181.8 s | 50000 samples | Mc=28.044  q=0.798


## 2. Run pocoMC (preconditioned SMC)

In [3]:
t_poco, s_poco = run('pocomc', dict(n_total=8000, n_effective=1500, n_active=500), 'fast_pocomc')
print(f'pocoMC: {t_poco:.1f} s | {s_poco.shape[0]} samples | '
      f'Mc={np.median(s_poco[:,0]):.3f}  q={np.median(s_poco[:,1]):.3f}')

> Running POCOMC sampling...


Iter: 0it [00:00, ?it/s]

Iter: 0it [00:00, ?it/s, beta=0, calls=0, ESS=1500, logZ=0, logP=0, acc=0, steps=0, eff=0]

Iter: 0it [00:00, ?it/s, beta=0, calls=500, ESS=1500, logZ=0, logP=-1.78e+4, acc=1, steps=1, eff=1]

Iter: 1it [00:00,  1.13it/s, beta=0, calls=500, ESS=1500, logZ=0, logP=-1.78e+4, acc=1, steps=1, eff=1]

...

Iter: 43it [16:55, 39.29s/it, beta=1, calls=368000, ESS=7552, logZ=-1.13e+4, logP=-1.13e+4, acc=0.846, steps=20, eff=0.93]

Iter: 43it [16:56, 39.29s/it, beta=1, calls=368500, ESS=7552, logZ=-1.13e+4, logP=-1.13e+4, acc=0.847, steps=21, eff=0.93]

Iter: 43it [17:13, 39.29s/it, beta=1, calls=372596, ESS=7552, logZ=-1.13e+4, logP=-1.13e+4, acc=0.847, steps=21, eff=0.93]

Iter: 43it [17:13, 24.03s/it, beta=1, calls=372596, ESS=7552, logZ=-1.13e+4, logP=-1.13e+4, acc=0.847, steps=21, eff=0.93]

> POCOMC sampling finished. Output at: results/pe_fast_nb/chains/POCO_fast_pocomc_pocomc.pkl
Loading samples from results/pe_fast_nb/chains/POCO_fast_pocomc_pocomc.pkl
pocoMC: 1033.8 s | 8674 samples | Mc=28.044  q=0.798


## 3. Compare posteriors and timing

Both samplers should recover the injected $(\mathcal{M}, q)$; the overlaid corner shows they
agree, and the timing table reports the cost of each.

In [4]:
labels = [r'$\mathcal{M}$', r'$q$']
rng = [(theta_true[0]-0.4, theta_true[0]+0.4), (theta_true[1]-0.25, min(1.0, theta_true[1]+0.25))]
fig = corner.corner(s_eryn[:, :2], labels=labels, color='#0f9b8e', truths=theta_true[:2],
                    range=rng, hist_kwargs={'density': True}, plot_datapoints=False)
corner.corner(s_poco[:, :2], fig=fig, color='#ca0147', range=rng,
              hist_kwargs={'density': True}, plot_datapoints=False)
fig.legend(handles=[plt.Line2D([], [], color='#0f9b8e', label='Eryn'),
                    plt.Line2D([], [], color='#ca0147', label='pocoMC')],
           loc='upper right', frameon=False)
out = os.path.join(OUT, 'fast_pe_corner.png')
fig.savefig(out, dpi=150, bbox_inches='tight')
print('saved ->', out)
print(f'\nTIMING:  Eryn {t_eryn:8.1f} s   |   pocoMC {t_poco:8.1f} s')

saved -> results/pe_fast_nb/fast_pe_corner.png

TIMING:  Eryn   2181.8 s   |   pocoMC   1033.8 s


## Takeaway

Two very different samplers — ensemble PT-MCMC (Eryn) and preconditioned SMC (pocoMC) — driven
through the **same** `LVKinference` interface and the **same** hyperbolic likelihood, recover the
injection consistently. Swap `--sampler` / the `sampler_name` argument to switch; scale
`examples/pe_full/bbh_full_pe.py` to the full 14-parameter problem on a cluster.